In [0]:
%sql
create schema if not exists dev.dlt
comment 'schema for delta live table demos'

In [0]:
%sql 
CREATE VOLUME IF NOT EXISTS dev.dlt.dlt_volume
comment 'volume for delta live table demos'

In [0]:
%sql
create table if not exists dev.dlt.orders_dlt deep clone samples.tpch.orders;
create table if not exists dev.dlt.customers_dlt deep clone samples.tpch.customer;

In [0]:
dbutils.fs.mkdirs('/Volumes/dev/dlt/dlt_volume/auto_loader/')
dbutils.fs.mkdirs('/Volumes/dev/dlt/dlt_volume/files/')

inserting data for incremental load


In [0]:
%sql
insert into dev.dlt.orders_dlt 
select * from samples.tpch.orders
limit 10000;

In [0]:
%sql
select * from dev.dlt.orders_dlt

In [0]:
%sql
select distinct o_orderstatus from dev.dlt.joined_silver

In [0]:
%sql
SELECT *
FROM event_log('100d630a-0d7a-4eb7-9bfd-d11a5f3a4091')
ORDER BY timestamp DESC;

for scd type 1 and type 2


In [0]:
%sql
alter table dev.dlt.customers_dlt
add columns (
  _src_action string,
  _src_insert_dt timestamp
)


In [0]:
%sql
update dev.dlt.customers_dlt
set _src_action = "I",
_src_insert_dt = current_timestamp() - interval '3 days'


In [0]:
%sql
select * from dev.dlt.customers_dlt limit 10

In [0]:
%sql
select * from dev.dlt.customer_scd1_bronze --where c_custkey = 6

In [0]:
%sql
select * from dev.dlt.customer_scd2_bronze where c_custkey = 6
order by `__START_AT`

In [0]:
%sql
INSERT INTO dev.dlt.customers_dlt
VALUES (
  6,
  'Customer#000412450',
  'fUD6IOGdtF',
  20,
  '30-293-696-5047',
  4406.28,
  'BUILDING',
  'New Changed Record',
  'I',
  current_timestamp()
);

for back filling


In [0]:
%sql
-- Insert changed record for cust_key 6 for back loading

INSERT INTO dev.dlt.customers_dlt
VALUES (
  6,
  'Customer#000412450',
  'fUD6IOGdtF',
  20,
  '30-293-696-5047',
  4406.28,
  'BUILDING',
  'Old Record for Back Loading',
  'I',
  current_timestamp() - interval 1 days
);

In [0]:
%sql
-- Insert changed record for cust_key 6

INSERT INTO dev.dlt.customers_dlt
VALUES (
  6,
  'Customer#000412450',
  'fUD6IOGdtF',
  20,
  '30-293-696-5047',
  4406.28,
  'BUILDING',
  'New Changed Record',
  'D',
  current_timestamp()
);

In [0]:
%sql
--Truncating records with T
INSERT INTO dev.dlt.customers_dlt
VALUES (
  6,
  'Customer#000412450',
  'fUD6IOGdtF',
  20,
  '30-293-696-5047',
  4406.28,
  'BUILDING',
  'New Changed Record',
  'T',
  current_timestamp()
);

data quality rules code


In [0]:
%sql
-- Insert data in Orders Raw
-- Setting o_orderstatus other than O, F or P
-- Setting o_orderprice > 0

INSERT INTO dev.dlt.orders_dlt
VALUES
  (99999, 227285, 'NA', 162169.66, '1995-10-11', '1-URGENT', 'Clerk#00000432', 0, 'Demo record 1 for Expectations test'),
  (99999, 227285, 'O', 100, '1995-10-11', '1-URGENT', 'Clerk#00000432', 0, 'Demo record 2 for Expectations test'),
  (99999, 227285, NULL, 999, '1995-10-11', '1-URGENT', 'Clerk#00000432', 0, 'Demo record 3 for Expectations test');

In [0]:
%sql
-- Insert data in Customer Raw
-- Setting c_mktsegment as null
INSERT INTO dev.dlt.customers_dlt
VALUES(
  9999, 
  'Customer#000412450', 
  'fUD6IoGdtF', 
  10, 
  '30-293-696-5047', 
  4406.28, 
  null, 
  'Demo record for Expectation test', 
  'I', 
  current_timestamp()
);

In [0]:
%sql
select * from dev.dlt.orders_union_bronze 
where o_orderkey = 9999;
